# k-mamba CUDA — 500M params sur Google Colab

Entraînement d'un modèle Mamba ~500M paramètres avec tokenizer BPE Rust.

**Config** : VOCAB=32K, DIM=1024, STATE=2048, LAYERS=24, SEQ=1024

**GPU** : Tesla T4 (16GB VRAM) — Runtime → Change runtime type → GPU

## 1. Vérification GPU

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Installation Rust + dépendances

In [ ]:
# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
!source $HOME/.cargo/env && rustc --version

# Install build tools
!apt-get update -qq && apt-get install -y -qq gcc g++ nasm git make

## 3. Clone et build k-mamba

In [ ]:
%cd /content
!rm -rf k-mamba
!git clone https://github.com/goldensam777/k-mamba.git
%cd k-mamba

In [ ]:
# Build tokenizer Rust
!source $HOME/.cargo/env && cd rust_tokenizer && cargo build --release 2>&1 | tail -5

In [ ]:
# Build k-mamba CUDA
!make clean
!make lib 2>&1 | tail -10
!make models/kmamba_cuda 2>&1 | tail -5

## 4. Configuration 500M params

In [ ]:
CONFIG = {
    'vocab_size': 32768,      # BPE 32K
    'dim': 1024,              # 1024 dimensions
    'state_size': 2048,       # 2048 états SSM
    'n_layers': 24,           # 24 blocs Mamba
    'seq_len': 1024,          # Contexte 1024 tokens
    'batch_size': 4,          # 4 séquences (limite T4 16GB)
    'total_params': '500M',   # ~500M paramètres
    'lr': 5e-4,               # Learning rate
    'total_tokens': 1_000_000_000,  # 1B tokens (ajuster selon besoin)
}

print(f"Modèle: {CONFIG['total_params']} paramètres")
print(f"Tokens cible: {CONFIG['total_tokens']:,}")
print(f"Batch: {CONFIG['batch_size']} × {CONFIG['seq_len']} = {CONFIG['batch_size'] * CONFIG['seq_len']:,} tokens/step")
print(f"VRAM estimée: ~12-14 GB")

## 5. Préparation données

In [ ]:
!mkdir -p data

# Option 1: Uploader fichier via panneau gauche (fichier) → /content/k-mamba/data/
# Option 2: Télécharger corpus
# !wget -q -O data/corpus.txt "URL"

!ls -lh data/

## 6. Entraînement

In [ ]:
!mkdir -p logs

# Lancer entraînement
!./models/kmamba_cuda train data/corpus.txt kmamba_500M.bin logs \
    --total-tokens {CONFIG['total_tokens']} 2>&1 | tee logs/train.log

## 7. Visualisation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    steps = pd.read_csv('logs/step.csv')
    epochs = pd.read_csv('logs/epoch.csv')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(steps['step'], steps['loss'])
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss par Step')
    axes[0].grid(True)
    
    axes[1].plot(epochs['epoch'], epochs['train_bt'], label='Train')
    axes[1].plot(epochs['epoch'], epochs['val'], label='Val')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Loss par Epoch')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Logs non disponibles: {e}")

## 8. Téléchargement checkpoint

In [ ]:
from google.colab import files

# Télécharger checkpoint
files.download('kmamba_500M.bin')

# Télécharger logs
!zip -r logs.zip logs/
files.download('logs.zip')